# Ideas for Machine Learning Improvements
I will develop the following Machine Learning Idea in this notebook
- 1) Develop a more sophisticated **Voting Ensemble**
- 2) Use SHAP instead of LIME

# 1) Improved Voting Ensemble:
In A3 I created a voting ensemble using 3 Decision Tree models. There are many ways to improve a voting ensemble, and I have picked the following two to develop:
- 1a: Increase number of models in Voting Ensemble
- 1b: Increase accuracy of models in Voting Ensemble

In [1]:
pip install scikit-learn

In [2]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")['train']
print(imdb_dataset[-1])

{'text': 'The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crocker) are highlights.', 'label': 1}


In [4]:
train_data = []
train_data_labels = []
for item in imdb_dataset:
  train_data.append(item['text'])
  train_data_labels.append(item['label'])
print(train_data[-1])
print(train_data_labels[-1])

The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crocker) are highlights.
1


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(analyzer='word',max_features=1000,lowercase=True,stop_words='english',ngram_range=(1,2))
features = vectorizer.fit_transform(train_data).toarray()

In [6]:
print(features.shape)
print(vectorizer.get_feature_names_out())

(25000, 1000)
['10' '15' '20' '30' '50' '80' '90' 'able' 'absolutely' 'accent' 'act'
 'acted' 'acting' 'action' 'actor' 'actors' 'actress' 'actual' 'actually'
 'add' 'admit' 'adult' 'adventure' 'age' 'ago' 'agree' 'air' 'amazing'
 'america' 'american' 'amusing' 'animated' 'animation' 'annoying' 'anti'
 'apart' 'apparently' 'appear' 'appears' 'appreciate' 'aren' 'art' 'aside'
 'ask' 'atmosphere' 'attempt' 'attempts' 'attention' 'audience'
 'audiences' 'average' 'avoid' 'away' 'awesome' 'awful' 'baby'
 'background' 'bad' 'bad movie' 'badly' 'band' 'barely' 'based' 'basic'
 'basically' 'battle' 'beautiful' 'beauty' 'begin' 'beginning' 'begins'
 'believable' 'believe' 'ben' 'best' 'better' 'big' 'biggest' 'bit'
 'bizarre' 'black' 'blood' 'body' 'book' 'books' 'bored' 'boring' 'box'
 'boy' 'boys' 'br' 'br 10' 'br br' 'br don' 'br film' 'br movie'
 'br story' 'brain' 'break' 'brilliant' 'bring' 'brings' 'british'
 'brother' 'brothers' 'brought' 'budget' 'bunch' 'business' 'buy' 'called'
 'ca

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(features,train_data_labels,train_size=0.9,random_state=123)

### Create Models
- This time I will create a voting ensemble using 5 models rather than 3
- The 2 models I will add are the 2 models with the highest accuracy, which we know from A1:
- Logistic Regression & Multinomial Naive Bayes

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

model_dt1 = DecisionTreeClassifier(criterion="log_loss")
model_dt2 = DecisionTreeClassifier(criterion="entropy")
model_dt3 = DecisionTreeClassifier(criterion="gini")
model_nb = MultinomialNB()
model_lr = LogisticRegression()

### Train models 
- Using the IMDB training dataset

In [9]:
model_dt1.fit(X_train, y_train)
model_dt2.fit(X_train, y_train)
model_dt3.fit(X_train, y_train)
model_nb.fit(X_train, y_train)
model_lr.fit(X_train, y_train)

C:\Users\humay\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

### Test Models' Accuracy

In [10]:
from sklearn.metrics import accuracy_score

y_pred_dt1 = model_dt1.predict(X_val)
y_pred_dt2 = model_dt1.predict(X_val)
y_pred_dt3 = model_dt3.predict(X_val)
y_pred_nb = model_nb.predict(X_val)
y_pred_lr = model_lr.predict(X_val)

print(f"DT1: {accuracy_score(y_pred_dt1, y_val)*100}%")
print(f"DT2: {accuracy_score(y_pred_dt2, y_val)*100}%")
print(f"DT3: {accuracy_score(y_pred_dt3, y_val)*100}%")
print(f"NB: {accuracy_score(y_pred_nb, y_val)*100}%")
print(f"LR: {accuracy_score(y_pred_lr, y_val)*100}%")

DT1: 68.92%
DT2: 68.92%
DT3: 70.24000000000001%
NB: 80.64%
LR: 83.28%


## 1a) Create Voting Ensemble with 5 Models
New models added: Multinomial Naive Bayes & Logistic Regression

**How the Voting Ensemble works**
- First create a vertical stack of the 5 predictions for a review in our validation set
- Then we go through each row which contains 5 predictions for a review (from the 5 models)
- We choose the result/class/label which is most frequent (appears at least twice)
- This label is then what we choose for our voting ensemble prediction

In [11]:
import numpy as np
# Compare accuracy of 3 models vs 5 models

# Previous Hard-Voting Voting Ensemble: 3 models
hv_pred_3models = np.vstack((y_pred_dt1, y_pred_dt2, y_pred_dt3)).T
hv_y_pred_ve_3models = np.array([np.bincount(i).argmax() for i in hv_pred_3models])


# New Hard-Voting Voting Ensemble: 5 models
hv_pred_5models = np.vstack((y_pred_dt1, y_pred_dt2, y_pred_dt3, y_pred_nb, y_pred_lr)).T
hv_y_pred_ve_5models = np.array([np.bincount(i).argmax() for i in hv_pred_5models])

### Accuracy of Both VEs

In [12]:
print(f"Old Hard VE: {accuracy_score(hv_y_pred_ve_3models, y_val)*100}%")
print(f"New Hard VE: {accuracy_score(hv_y_pred_ve_5models, y_val)*100}%")

Old Hard VE: 68.92%
New Hard VE: 76.72%


We see that the Voting Ensemble's Accuracy has improved with 5 models

## 1b) Improve Individual Models' Accuracies

**How to Improve Accuracy**:
- Use TF-IDF Vectorizer rather than CountVectorizer: The TF-IDF vectorizer is better than CountVectorizer as it weighs the importance of words rather than just having a simple count of how often they appear.
- Increase max_features from 1000 to 2500: top 2500 most frequent words

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_new = TfidfVectorizer(analyzer='word',max_features=2500,lowercase=True,stop_words='english',ngram_range=(1,2))
features_new = vectorizer_new.fit_transform(train_data).toarray()

X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(features_new,train_data_labels,train_size=0.9,random_state=123)

new_model_dt1 = DecisionTreeClassifier(criterion="log_loss")
new_model_dt2 = DecisionTreeClassifier(criterion="entropy")
new_model_dt3 = DecisionTreeClassifier(criterion="gini")
new_model_nb = MultinomialNB()
new_model_lr = LogisticRegression()

new_model_dt1.fit(X_train_new, y_train_new)
new_model_dt2.fit(X_train_new, y_train_new)
new_model_dt3.fit(X_train_new, y_train_new)
new_model_nb.fit(X_train_new, y_train_new)
new_model_lr.fit(X_train_new, y_train_new)

LogisticRegression()

### Test New Individual Models' Accuracies

In [14]:
new_y_pred_dt1 = new_model_dt1.predict(X_val_new)
new_y_pred_dt2 = new_model_dt1.predict(X_val_new)
new_y_pred_dt3 = new_model_dt3.predict(X_val_new)
new_y_pred_nb = new_model_nb.predict(X_val_new)
new_y_pred_lr = new_model_lr.predict(X_val_new)

print(f"Old DT1: {accuracy_score(y_pred_dt1, y_val)*100}% | New DT1: {accuracy_score(new_y_pred_dt1, y_val)*100}%")
print(f"Old DT2: {accuracy_score(y_pred_dt2, y_val)*100}% | New DT2: {accuracy_score(new_y_pred_dt2, y_val)*100}%")
print(f"Old DT3: {accuracy_score(y_pred_dt3, y_val)*100}% | New DT3: {accuracy_score(new_y_pred_dt3, y_val)*100}%")
print(f"Old NB: {accuracy_score(y_pred_nb, y_val)*100}% | New NB: {accuracy_score(new_y_pred_nb, y_val)*100}%")
print(f"Old LR: {accuracy_score(y_pred_lr, y_val)*100}% | New LR: {accuracy_score(new_y_pred_lr, y_val)*100}%")

Old DT1: 68.92% | New DT1: 70.56%
Old DT2: 68.92% | New DT2: 70.56%
Old DT3: 70.24000000000001% | New DT3: 70.04%
Old NB: 80.64% | New NB: 83.76%
Old LR: 83.28% | New LR: 86.0%


OK, so we see that 4/5 models have increased their accuracy. The only model which decreased accuracy is: Decision Tree (gini).

**Why DT (Gini) decreased accuracy:**
- Gini Impurity less effective on TF-IDF: meaningful class distinctions may work better for  counts (CountVectorizer) rather than weights (TF-IDF)

- Decision Trees tend to overfit on data. Gini Imurity measures the purity of a "split", which his harder to do with overfitted models

Therefore for just DT (gini) we will keep the old model, but for all other models we will use the new improved versions

### Check Voting Ensemble with New Individual Models

In [15]:
new_hv_pred_5models = np.vstack((new_y_pred_dt1, new_y_pred_dt2, y_pred_dt3, new_y_pred_nb, y_pred_lr)).T
new_hv_y_pred_ve_5models = np.array([np.bincount(i).argmax() for i in hv_pred_5models])

print(f"Old 5 model Hard VE: {accuracy_score(hv_y_pred_ve_5models, y_val)*100}%")
print(f"New 5 model Hard VE: {accuracy_score(new_hv_y_pred_ve_5models, y_val)*100}%")

Old 5 model Hard VE: 76.72%
New 5 model Hard VE: 76.72%


Accuracy remains the same, even after improving the accuracy of our individual models.

This means that although the individual models predicted better results they couldn’t have been for the same reviews, as the other models would outvote them.

In [16]:
from sklearn.metrics import confusion_matrix

old_ve_cm = confusion_matrix(y_val, hv_y_pred_ve_5models, labels=[0,1])
print(old_ve_cm)

new_ve_cm = confusion_matrix(y_val, new_hv_y_pred_ve_5models, labels=[0,1])
print(new_ve_cm)

[[949 292]
 [290 969]]
[[949 292]
 [290 969]]


From a look at the confusion matrics of both Voting Ensembles we see that it makes the exact same predictions. This means that although we slightly increased some models' accuracies by a few % the Voting Ensemble still ended up making the same decisions.

We will use the new Voting Ensemble in our Voting Ensemble Function

# Voting Ensemble Function

In [54]:
def voting_ensemble(data):
    
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.linear_model import LogisticRegression
    from datasets import load_dataset
    
    imdb_dataset = load_dataset("imdb")['train']
    train_data = []
    train_data_labels = []
    for item in imdb_dataset:
        train_data.append(item['text'])
        train_data_labels.append(item['label'])
    
    vectorizer = CountVectorizer(analyzer='word',max_features=1000,lowercase=True,stop_words='english',ngram_range=(1,2))
    features = vectorizer.fit_transform(train_data).toarray()
    
    vectorizer_new = TfidfVectorizer(analyzer='word',max_features=2500,lowercase=True,stop_words='english',ngram_range=(1,2))
    features_new = vectorizer_new.fit_transform(train_data).toarray()
    
    X_train, X_val, y_train, y_val = train_test_split(features,train_data_labels,train_size=0.9,random_state=123)
    X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(features_new,train_data_labels,train_size=0.9,random_state=123)

    new_model_dt1 = DecisionTreeClassifier(criterion="log_loss")
    new_model_dt2 = DecisionTreeClassifier(criterion="entropy")
    new_model_dt3 = DecisionTreeClassifier(criterion="gini")
    new_model_nb = MultinomialNB()
    new_model_lr = LogisticRegression()
    
    new_model_dt1.fit(X_train_new, y_train_new)
    new_model_dt2.fit(X_train_new, y_train_new)
    model_dt3.fit(X_train, y_train)              # old model accuracy is better
    new_model_nb.fit(X_train_new, y_train_new)
    new_model_lr.fit(X_train_new, y_train_new)
    
    data_1000 = vectorizer.transform(data).toarray()  # For Decision Tree 3
    data_2500 = vectorizer_new.transform(data).toarray()  # For all other models
    
    new_y_pred_dt1 = new_model_dt1.predict(data_2500)
    new_y_pred_dt2 = new_model_dt1.predict(data_2500)
    y_pred_dt3 = model_dt3.predict(data_1000)
    new_y_pred_nb = new_model_nb.predict(data_2500)
    new_y_pred_lr = new_model_lr.predict(data_2500)
        
    pred_ve = np.vstack((new_y_pred_dt1, new_y_pred_dt2, y_pred_dt3, new_y_pred_nb, new_y_pred_lr)).T
    voting_ensemble_y_pred= np.array([np.bincount(i).argmax() for i in pred_ve])
    
    return voting_ensemble_y_pred

This is a large Voting Ensemble function with everything needed (excecpt for reviews), so that hypothetically some one can get predictions using just their review data without having anything else already set up.

Downside of a large Voting Ensmble function is it takes longer to get predictions than say if a user already has the individual models fitted. But this is more practical as it allows someone to get predictions with just their data and not need anything else.

## Test Voting Ensmble Function

In [1]:
import pandas as pd

dune = ["Watching this in IMAX is the correct way to watch. That is what the movies were made for. Incredible",
        "This is truly the cinematic event of our generation… don’t take it for granted",
        "this is #1. my favorite movie. it led me to paradise."]

megalopoplis = ["""Doesn't know what it should be. Cool idea but it feels like the end product doesn't match the vision. 
Some of the cinematography was brilliant while some scenes completely contradict that point. 
As the film progresses the obvious green screen becomes hard to avoid, which probably comes down to budgeting issues. 
Sometimes there's several plots going on at once, with none that are really good enough to really catch you attention, 
making it difficult to understand whats actually happening. 
Positive points are that there are some really nice shots & the world seems interesting to explore and understand. 
Glad to have watched because it's definitely an experience.""",
                "WORST movie i have ever seen.", 
                "most nonsensical and absurd movie i ever saw, unpleasant and annoying"]

dune_df = pd.DataFrame({'text': dune, 'label': [1, 1, 1]}, index=[0, 1, 2])
megalopoplis_df = pd.DataFrame({'text': megalopoplis, 'label': [0, 0, 0]}, index=[3, 4, 5])

test1 = pd.concat([dune_df, megalopoplis_df])

layer_cake = ["Action movie with a plot that keeps you there, confused but decent nonetheless", 
            "The budget of Layer Cake was 6.5 million. It looked like it was shot for 60 million dollars.", 
            "that yellow range rover is so ugly, yet so cool"]

jack_and_jill = ["Adam Sandler has blackmail on all of Hollywood, doesn't he", 
                 "I enjoyed none of this", 
                 "I thank my lucky stars I was unconscious for most of this"]

layer_cake_df = pd.DataFrame({'text': layer_cake, 'label': [1, 1, 1]}, index=[0, 1, 2])
jack_and_jill_df = pd.DataFrame({'text': jack_and_jill, 'label': [0, 0, 0]}, index=[3, 4, 5])

test2 = pd.concat([layer_cake_df, jack_and_jill_df])

In [2]:
#test1.to_csv("easy_test_sets.csv", index=False)
#test2.to_csv("hard_test_sets.csv", index=False)

In [56]:
test1_text = test1["text"]
test1_labels = test1["label"]

test1_y_pred = voting_ensemble(test1_text)

In [59]:
print(f"Voting Ensemble Accuracy (Test1): {accuracy_score(test1_y_pred, test1_labels)*100}%")

Voting Ensemble Accuracy (Test1): 83.33333333333334%


# Implement Voting Ensemble into FLASK Web Page
Dump models & vectorizer into pickle files. Implement voting ensemble in src/flask/app.py

In [2]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pickle

imdb_dataset = load_dataset("imdb")['train']
train_data = [item['text'] for item in imdb_dataset]
train_labels = [item['label'] for item in imdb_dataset]

vectorizer = TfidfVectorizer(analyzer='word', max_features=2500, lowercase=True, stop_words='english', ngram_range=(1, 2))
features = vectorizer.fit_transform(train_data).toarray()

X_train, X_val, y_train, y_val = train_test_split(features, train_labels, train_size=0.9, random_state=123)

# Train models
dt1 = DecisionTreeClassifier(criterion="log_loss").fit(X_train, y_train)
dt2 = DecisionTreeClassifier(criterion="entropy").fit(X_train, y_train)
dt3 = DecisionTreeClassifier(criterion="gini").fit(X_train, y_train)
nb = MultinomialNB().fit(X_train, y_train)
lr = LogisticRegression().fit(X_train, y_train)

models = {
    "dt1": dt1,
    "dt2": dt2,
    "dt3": dt3,
    "nb": nb,
    "lr": lr
}

# dump models and features into pickle files 
pickle.dump(models, open('models.pkl', 'wb'))
pickle.dump(vectorizer, open('features.pkl', 'wb'))